# Qwen-Image on Kaggle: 2 x Tesla T4 + Gradio

Run Cells 1-9 in order on every fresh Kaggle session. The Hugging Face cache and all model weights stay under `/kaggle/tmp/hf`; only the small cloned UI repository stays under `/kaggle/working`.

In [ ]:
# Cell 1: install the Kaggle runtime dependencies.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "git+https://github.com/huggingface/diffusers.git",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "gguf",
        "huggingface_hub",
        "hf_transfer",
        "gradio",
        "starlette<1.0.0",
        "google-cloud-bigquery-storage",
        "protobuf<7.0.0",
    ],
    check=True,
)

In [ ]:
# Cell 2: configure the large ephemeral cache, then clone only the small UI source.
import os
import subprocess
from pathlib import Path

HF_HOME = "/kaggle/tmp/hf"
os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = f"{HF_HOME}/hub"
os.environ["HF_XET_CACHE"] = f"{HF_HOME}/xet"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

repo_dir = Path("/kaggle/working/Qwen-Image-Test")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Ron-yos58/Qwen-Image-Test.git", str(repo_dir)],
        check=True,
    )
print(f"HF_HOME: {HF_HOME}")
print(f"UI source: {repo_dir}")


In [ ]:
# Cell 3: stop early unless Kaggle exposed exactly two CUDA GPUs.
import subprocess
import torch

assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator first."
assert torch.cuda.device_count() == 2, (
    f"Expected 2 GPUs, found {torch.cuda.device_count()}. Select 2x Tesla T4 in Kaggle settings."
)
for device_index in range(torch.cuda.device_count()):
    print(f"GPU {device_index}: {torch.cuda.get_device_name(device_index)}")
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
# Cell 4: download the quantized transformer into HF_HOME, never /kaggle/working.
from huggingface_hub import hf_hub_download

GGUF_REPO = "city96/Qwen-Image-gguf"
GGUF_FILENAME = "qwen-image-Q4_K_M.gguf"
gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILENAME)
print(f"GGUF checkpoint: {gguf_path}")


In [ ]:
# Cell 5: load Q4_K_M in FP16 compute mode directly on GPU 0, without .to("cuda").
from diffusers import GGUFQuantizationConfig, QwenImageTransformer2DModel

DTYPE = torch.float16
MAX_MEMORY = {0: "14GiB", 1: "14GiB"}
gguf_quantization_config = GGUFQuantizationConfig(compute_dtype=DTYPE)

# Passed components are not re-dispatched by pipeline.from_pretrained, so load this GGUF once on GPU 0.
transformer = QwenImageTransformer2DModel.from_single_file(
    gguf_path,
    config="Qwen/Qwen-Image",
    subfolder="transformer",
    quantization_config=gguf_quantization_config,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device="cuda:0",
)
print("Transformer placement:", getattr(transformer, "hf_device_map", {"": "cuda:0"}))

In [ ]:
# Cell 6: load the 8-bit text encoder on GPU 1, then assemble the FP16 Qwen pipeline.
from diffusers import QwenImagePipeline
from diffusers.quantizers import PipelineQuantizationConfig
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

text_encoder_bnb_config = BitsAndBytesConfig(load_in_8bit=True)
text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen-Image",
    subfolder="text_encoder",
    quantization_config=text_encoder_bnb_config,
    torch_dtype=DTYPE,
    device_map={"": 1},
    max_memory=MAX_MEMORY,
    low_cpu_mem_usage=True,
)

# Current Diffusers requires a pipeline wrapper; it preserves the requested 8-bit config for text_encoder.
pipeline_quantization_config = PipelineQuantizationConfig(
    quant_mapping={"text_encoder": text_encoder_bnb_config}
)
pipe = QwenImagePipeline.from_pretrained(
    "Qwen/Qwen-Image",
    transformer=transformer,
    text_encoder=text_encoder,
    torch_dtype=DTYPE,
    quantization_config=pipeline_quantization_config,
    device_map="balanced",
    max_memory=MAX_MEMORY,
    low_cpu_mem_usage=True,
)
print("Pipeline device map:", getattr(pipe, "hf_device_map", None))
print("Text encoder placement:", getattr(pipe.text_encoder, "hf_device_map", {"": "cuda:1"}))

In [ ]:
# Cell 7: inspect placement and VRAM; both GPUs must show nonzero usage in nvidia-smi.
import subprocess

print("Transformer map:", getattr(pipe.transformer, "hf_device_map", {"": "cuda:0"}))
print("Text encoder map:", getattr(pipe.text_encoder, "hf_device_map", {"": "cuda:1"}))
print("Pipeline map:", getattr(pipe, "hf_device_map", None))
for device_index in range(2):
    allocated_gib = torch.cuda.memory_allocated(device_index) / 1024**3
    reserved_gib = torch.cuda.memory_reserved(device_index) / 1024**3
    print(f"GPU {device_index}: allocated={allocated_gib:.2f} GiB, reserved={reserved_gib:.2f} GiB")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total",
        "--format=csv,noheader,nounits",
    ],
    check=True,
)

In [ ]:
# Cell 8: replace the Spaces app.py with a Gradio UI that only uses the already loaded global pipe.
import importlib.util
import sys

APP_SOURCE = r'''
import random

import gradio as gr
import torch

MAX_SEED = 2_147_483_647
PIPE = None


def set_pipeline(pipeline):
    global PIPE
    PIPE = pipeline


def _valid_dimension(value):
    value = max(640, min(1328, int(value)))
    return value - (value % 16)


def generate(prompt, negative_prompt, steps, cfg_scale, width, height, seed, randomize_seed):
    if PIPE is None:
        raise gr.Error("Pipeline is not loaded. Run Cells 1-7 first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Prompt is required.")

    actual_seed = random.randint(0, MAX_SEED) if randomize_seed else int(seed)
    actual_negative_prompt = negative_prompt if negative_prompt and negative_prompt.strip() else " "
    generator = torch.Generator("cpu").manual_seed(actual_seed)

    with torch.inference_mode():
        image = PIPE(
            prompt=prompt,
            negative_prompt=actual_negative_prompt,
            num_inference_steps=int(steps),
            true_cfg_scale=float(cfg_scale),
            width=_valid_dimension(width),
            height=_valid_dimension(height),
            generator=generator,
            max_sequence_length=512,
        ).images[0]
    return image, actual_seed


def build_demo():
    with gr.Blocks(title="Qwen-Image on Kaggle") as demo:
        gr.Markdown("# Qwen-Image")
        with gr.Row():
            with gr.Column():
                prompt = gr.Textbox(label="Prompt", lines=5)
                negative_prompt = gr.Textbox(label="Negative prompt", value=" ", lines=2)
                with gr.Row():
                    steps = gr.Slider(20, 50, value=40, step=1, label="Steps")
                    cfg_scale = gr.Slider(1, 8, value=4, step=0.5, label="True CFG scale")
                with gr.Row():
                    width = gr.Slider(640, 1328, value=1024, step=16, label="Width")
                    height = gr.Slider(640, 1328, value=1024, step=16, label="Height")
                seed = gr.Number(value=0, precision=0, minimum=0, maximum=MAX_SEED, label="Seed")
                randomize_seed = gr.Checkbox(value=True, label="Randomize seed")
                generate_button = gr.Button("Generate", variant="primary")
            with gr.Column():
                image = gr.Image(label="Generated image", type="pil")
                used_seed = gr.Number(label="Used seed", precision=0)

        generate_button.click(
            fn=generate,
            inputs=[prompt, negative_prompt, steps, cfg_scale, width, height, seed, randomize_seed],
            outputs=[image, used_seed],
        )
    return demo
'''

app_path = repo_dir / "app.py"
app_path.write_text(APP_SOURCE, encoding="utf-8")
app_spec = importlib.util.spec_from_file_location("kaggle_qwen_app", app_path)
kaggle_app = importlib.util.module_from_spec(app_spec)
sys.modules[app_spec.name] = kaggle_app
app_spec.loader.exec_module(kaggle_app)
kaggle_app.set_pipeline(pipe)
demo = kaggle_app.build_demo()
print(f"Kaggle UI written to: {app_path}")

In [ ]:
# Cell 9: run this last; it blocks while Gradio prints the public https://xxxx.gradio.live URL.
demo.queue().launch(share=True, show_error=True)